🔰PyTorchでニューラルネットワーク基礎　#41 【GPT編・事前学習２】



### 内容
* Qiitaの記事と連動しています
* 各種ファイルの保存先は環境によって適宜変更してください
* pytorchの「TransformerEncoderLayer+因果マスク」を使わない小型GPTタイプの実装

### データについて
* Livedoorニュースコーパスのlivedoor-itカテゴリを利用します。
* huggingfaceの　llm-book/livedoor-news-corpus　などから適宜ダウンロードしてください。
* 先頭から3行のURL、日付、タイトルを削除したものを利用します。

### トークナイザーについて
* tokenizer/livedoor_tokenizer_10k.json
    * bytelevel BPEで構成した語彙数10kのtokenizer


### 今回扱う内容
1. GPTタイプの事前学習
2. Transformerブロックの作成
3. 日本語データでの学習

### 注意点
* 汎用的な内容は生成不能
* 学習した内容、学習した文と類似文章が入力されるとうまく生成される
* 学習していない内容は、全くだめ
* モデルサイズ・データサイズが小さいので完全コピーに近い生成がみられるがこれが正常な状況
* 系列長を256とある程度長くしないと、指示チューニング時にトークン不足となる
* 事前学習だけを考える場合、GPUメモリとの兼ね合いで seq_len=128でも事前学習による生成可能

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from pathlib import Path
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{device=}")

device=device(type='cuda')


In [2]:
data_dir = "./data/it_texts/"                                  # 学習用データのディレクトリ
tokenizer_filename = "tokenizer/livedoor_tokenizer_10k.json"   # tokenizer
pretrain_filename = "model/it_seq_256_bpe_10k.model"

# トークナイザー
tokenizer = Tokenizer.from_file(tokenizer_filename)
print(f"size: {tokenizer.get_vocab_size()}")

size: 10000


In [3]:
class ModelConfig:
    def __init__(self, tokenizer):
        # モデル構造
        self.vocab_size = tokenizer.get_vocab_size()
        self.seq_len = 256   # 128トークンだとSFT時に少ない
        self.d_model = 256   # 512
        self.nhead = 8
        self.dim_feedforward = 4*self.d_model
        self.num_layers = 6
        self.dropout = 0.1
        
        # 特殊トークンID
        self.pad_token_id = tokenizer.token_to_id("<pad>")
        self.eod_token_id = tokenizer.token_to_id("<eod>")

        # 学習データに関する設定
        self.context_size = self.seq_len         # 学習できる長さ

        # 学習設定
        self.batch_size = 256
        self.learning_rate = 0.001  # これだとデフォルトと変わらない
        self.num_steps = 5000
        self.max_grad_norm = 1.0
        self.ignore_index = -100
        self.weight_decay = 0.1 # defalut:0.01 llama2: 0.1

    # 属性を追加・更新するメソッド
    # 設定時のタイポに注意だぞ〜😱〜
    def update(self, **kwargs):
        """渡されたキーワード引数で設定を動的に追加・更新する"""
        for key, value in kwargs.items():
            setattr(self, key, value)

## Transformerブロックの構成

In [4]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.d_model % config.nhead == 0
        self.nhead = config.nhead
        self.head_dim = config.d_model // config.nhead

        # q,k,v をまとめて1回のmatmulで　第22.5回風
        self.qkv = nn.Linear(config.d_model, 3 * config.d_model, bias=False)
        self.proj = nn.Linear(config.d_model, config.d_model, bias=False)
        self.attn_dropout_p = config.dropout
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        bs, seq_len, d_model = x.shape
        q, k, v = self.qkv(x).split(d_model, dim=2)
        # (bs, seq_len, d_model -> (bs, nhead, seq_len, head_dim)
        q = q.view(bs, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        k = k.view(bs, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        v = v.view(bs, seq_len, self.nhead, self.head_dim).transpose(1, 2)

        # PyTorchのScaled Dot Product Attention
        # 条件が合えばFlashAttentionなどの高速実装が自動選択される
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.attn_dropout_p if self.training else 0.0,
            is_causal=True,
        )

        y = y.transpose(1, 2).contiguous().view(bs, seq_len, d_model)
        return self.resid_dropout(self.proj(y))


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.fc1 = nn.Linear(config.d_model, config.dim_feedforward, bias=False)
        self.act = nn.GELU(approximate="tanh")
        self.fc2 = nn.Linear(config.dim_feedforward, config.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        return self.dropout(self.fc2(self.act(self.fc1(x))))


class TransformerBlock(nn.Module):
    """Pre-Normalization方式 """
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))     # Attention(LayerNorm(x)) + x
        x = x + self.mlp(self.ln2(x))      # MLP(LayerNorm(x)) + x
        return x

In [5]:
class DNN(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        # 埋め込み層
        self.token_embedding = nn.Embedding(num_embeddings=config.vocab_size, embedding_dim=config.d_model)
        self.pos_embedding = nn.Embedding(num_embeddings=config.seq_len, embedding_dim=config.d_model)
        self.dropout = nn.Dropout(config.dropout)

        # causal transformer layer
        self.transformer_blocks = nn.Sequential(*[TransformerBlock(config) for _ in range(config.num_layers)])

        # 最後の出力に向けた正規化とFC層　最終的に単語数になる
        self.layer_norm = nn.LayerNorm(config.d_model)
        self.fc = nn.Linear(config.d_model, config.vocab_size,bias=False)
        self.apply(self._init_weights)                    # 個別に初期化
        self.fc.weight = self.token_embedding.weight      # 重み共有
        

        # 残差方向の出力層だけ 1/sqrt(2N) スケール（GPT-2 の作法、深いと効きます）
        std = 0.02 / math.sqrt(2 * config.num_layers)
        for pn, p in self.named_parameters():
            if pn.endswith("proj.weight") or pn.endswith("fc2.weight"):
                nn.init.normal_(p, mean=0.0, std=std)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:          # fc は bias=False なのでスキップ
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)


    def forward(self, x):
        bs, seq_len = x.shape      
        positions = torch.arange(seq_len,device=x.device)
        
        tok_emb = self.token_embedding(x)
        pos_emb = self.pos_embedding(positions).unsqueeze(0)
        x = tok_emb + pos_emb           # トークン埋め込み＋位置埋め込み
        x = self.dropout(x)        
        x = self.transformer_blocks(x)  # 自己回帰型 transformer
        x = self.layer_norm(x)          # Layer正規化
        logits = self.fc(x)             # NPT: 次のトークン予測
        return logits

In [6]:
config = ModelConfig(tokenizer)
model = DNN(config).to(device)

In [7]:
from torchinfo import summary

summary(
    model,
    input_size=(1, config.seq_len),        # (batch, seq_len)
    dtypes=[torch.long],                   # 埋め込みの入力は整数インデックス
    depth=2,
)

Layer (type:depth-idx)                        Output Shape              Param #
DNN                                           [1, 256, 10000]           --
├─Embedding: 1-1                              [1, 256, 256]             2,560,000
├─Embedding: 1-2                              [256, 256]                65,536
├─Dropout: 1-3                                [1, 256, 256]             --
├─Sequential: 1-4                             [1, 256, 256]             --
│    └─TransformerBlock: 2-1                  [1, 256, 256]             787,456
│    └─TransformerBlock: 2-2                  [1, 256, 256]             787,456
│    └─TransformerBlock: 2-3                  [1, 256, 256]             787,456
│    └─TransformerBlock: 2-4                  [1, 256, 256]             787,456
│    └─TransformerBlock: 2-5                  [1, 256, 256]             787,456
│    └─TransformerBlock: 2-6                  [1, 256, 256]             787,456
├─LayerNorm: 1-5                              [1, 256,

### parameter数の確認
* model.parameters()を数えるだけ

In [8]:
unique = sum(p.numel() for p in model.parameters())
total  = sum(p.numel() for _, p in model.named_parameters(remove_duplicate=False))
print(f"unique={unique:,}  reported={total:,}  tied={total-unique:,}")

unique=7,350,784  reported=9,910,784  tied=2,560,000


In [9]:
# 重み共有されているかな？
model.fc.weight is model.token_embedding.weight  # True なら共有

True

In [10]:
class RandomGPTDataset(Dataset):
    def __init__(
        self,
        data_dir,
        tokenizer,
        context_size=128,
        samples_per_epoch=10_000,
    ):
        self.context_size = context_size
        self.samples_per_epoch = samples_per_epoch
        eod_id = tokenizer.token_to_id("<eod>")
        filenames = sorted(Path(data_dir).glob("*.txt"))

        all_ids = []
        for filename in filenames:
            text = filename.read_text(encoding="utf-8")
            document_ids = tokenizer.encode(text, add_special_tokens=False).ids
            all_ids.extend(document_ids)
            all_ids.append(eod_id)

        self.ids = torch.tensor(all_ids, dtype=torch.long)

    def __len__(self):
        # 1 epochあたりに何個のランダム窓を学習するか
        # 10,000個（回？）がデフォルト値
        return self.samples_per_epoch

    def __getitem__(self, idx):
        # idxは使わず、毎回ランダムな開始位置を選ぶ
        start = torch.randint(
            low=0,
            high=len(self.ids) - self.context_size,
            size=(1,),
        ).item()

        x = self.ids[start:start + self.context_size]
        y = self.ids[start + 1:start + self.context_size + 1]

        return {"ids": x, "labels": y}

In [11]:
dataset = RandomGPTDataset(
    data_dir=data_dir,
    tokenizer=tokenizer,
    context_size=config.context_size,
    samples_per_epoch=5_000,   # 10_000個のほうがいいけどとりあえず動作チェック
)

dataloader = DataLoader(
    dataset=dataset,
    batch_size=config.batch_size,  # メモリ足りない場合は小さくする
    shuffle=False,
    num_workers=0, # 2,4のほうが早い？要検討
    pin_memory=True,
)


In [12]:
# weight decayの設定
decay, no_decay = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if param.dim() >= 2:
        decay.append(param)      # Linear、Embeddingの重みに減衰追加
    else:
        no_decay.append(param)   # LayerNormのgain。これは weight_decay=0へ

optim_groups = [
    {"params": decay,    "weight_decay": config.weight_decay},
    {"params": no_decay, "weight_decay": 0.0},
]

optimizer = torch.optim.AdamW(optim_groups, lr=config.learning_rate, betas=(0.9, 0.95), fused=(device.type == "cuda"))
criterion = nn.CrossEntropyLoss(ignore_index=config.ignore_index)    # 今回ignore_indexを利用していないがSFT用につけておく

In [13]:
# step数で管理するためのデータローダー関数
def infinite_loader(dataloader):
    while True:
        for batch in dataloader:
            yield batch

data_iter = infinite_loader(dataloader)  # epochではなく、step数で計測

In [ ]:
from tqdm import tqdm
max_iters = config.num_steps  # step数
total_token = 0               # 累積トークン数を初期化
pbar = tqdm(range(max_iters))
model.train()                 # trainモードを明示
for step in pbar:
    batch = next(data_iter)
    input_ids = batch["ids"].to(device, non_blocking=True)
    labels = batch["labels"].to(device, non_blocking=True)
    optimizer.zero_grad()
    # flash attentionへ
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
        outputs = model(input_ids)
        loss = criterion(
            outputs.view(-1, config.vocab_size),
            labels.view(-1),
        ) 
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.max_grad_norm)  # 勾配クリップ
    optimizer.step()

    # このstepのトークン数 （バッチサイズ×系列長）を加算
    total_token += input_ids.shape[0] * input_ids.shape[1]

    # set_postfixは辞書が引数になる
    pbar.set_postfix({"loss": f"{loss.item():.4f}", "tokens": f"{total_token/1e6:.2f}M"})
    if (step+1) % 500 == 0:
        tqdm.write(f"{step+1}-step:\tloss:{loss.item():.3f}\ttokens:{total_token:,}")    

### 文章生成
* greedyアルゴリズムで文章生成

In [15]:
@torch.inference_mode()
def generate_text(
    model,
    input_ids,
    max_new_tokens=config.seq_len,
    eos_token_id=None,
):
    model.eval()
    generated = input_ids.to(device)
    seq_len = model.config.seq_len

    # greedyで単純に生成
    # eos_token_idで生成終了
    for _ in range(max_new_tokens):
        logits = model(generated[:, -seq_len:])
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)

        if eos_token_id is not None and next_token.item() == eos_token_id:
            break

    return generated

In [16]:
from tokenizers import decoders           # ByteLevelの時は利用する
tokenizer.decoder = decoders.ByteLevel()


def response(prompt):
    model.eval()
    model_input = tokenizer.encode(prompt, add_special_tokens=False)
    input_ids = torch.tensor([model_input.ids], dtype=torch.long, device=device)

    output_ids = generate_text(
        model=model,
        input_ids=input_ids,
        max_new_tokens=config.seq_len,
        eos_token_id=config.eod_token_id,
    )

    # 全文をデコード
    generated_text = tokenizer.decode(output_ids[0].tolist(), skip_special_tokens=False)
    print(generated_text)


### 続く文章の生成
* it_prompt.csvに記載した文章の続きを表示してみる

In [17]:
import pandas as pd

df = pd.read_csv("./data/it_prompt.csv")

for num, prompt in enumerate(df["prompt"]):
    print(f"【{num+1}番目】")
    response(prompt)

【1番目】
iPadをはじめとするタブレット端末が市場で大きなブレイクを遂げると言われています。タブレットは手持ちで使える点が大きなメリットですが、長時間手で持っているのはやはり疲れてしまいます。

家庭でキッチンやリビングなど、家事をしながら、あるいはカラオケパーティーでタブレットのカラオケアプリを楽しんだり、電子書籍を読んだりする際に、立ったままタブレットを使えたら、どれほど便利かと思うのではないでしょうか。

これまで「イケショップのレア物」では、タブレットやPCで使える便利なアイテムを紹介してきましたが、立ったままで使用するためのグッズは存在しませんでした。

今回ご紹介するのは、なんと、立ったままタブレット端末が使える「Tablet PC Tripod Stand」です。待ちに待った製品について、どのようなものなのかを詳しくご紹介します。

**しっかり固定し、自由な角度で調整可能**
「Tablet PC Tripod Stand」は、タブレット端末を傍らに置くのに便利なスタンドです。上下からしっかりはさみ込むホルダー部分は、自由に角度を変えられるため、最適なポジションを簡単に調節できます。ホルダーはボールタイプのジョイントになっているため、非常に自由な角度での使用が可能です。

**多様なデバイスに対応
【2番目】
Googleロゴのアニメーションを観察するとオシロスコープの波形のようなものが表示されます。このロゴをクリックすると、「ハインリヒヘルツ」という言葉が検索されるのですが、これはハインリヒ・ルドルフ・ヘルツ氏の誕生日を記念しているためです。

ハインリヒ・ルドルフ・ヘルツ氏はドイツの物理学者であり、マックスウェル氏の電磁気理論をさらに明確化し発展させました。彼は1888年に、電磁波の放射の存在を、それを生成・検出する機械の構築によって初めて実証した人物です。

彼の功績により、彼の名前は周波数を示すSI単位である「ヘルツ」に採用されました。

つまり、Googleロゴのアニメーションは、周波数の波を表現したものだったのです。

このエピソードに興味を持たれた方は、ぜひインターネットで検索を試みてみてはいかがでしょうか。<eod>
【3番目】
HPのUltrabookラインナップでは、その全サイズ、13.3型液晶ディスプレイを搭載しています。


### モデルの保存

In [ ]:
torch.save({
        "model_state_dict": model.state_dict(),
        "config": config.__dict__,  # configも一緒に保存
        }, pretrain_filename)

In [14]:
checkpoint = torch.load(pretrain_filename)
config = ModelConfig(tokenizer)
config.__dict__.update(checkpoint["config"])
model = DNN(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>